# 05. XGBoost Baseline Model
**ML-Powered Intrusion Detection System (IDS) for Secure Network Monitoring**

This notebook trains and evaluates the XGBoost gradient-boosted decision trees baseline and compares its performance against the Random Forest baseline.

## 1. Imports & Configuration

In [ ]:
import sys
import json
import time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score

# Ensure project root is in sys.path
ROOT_DIR = Path("..").resolve()
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

from src.models.xgboost_model import XGBoostBaselineModel
from src.preprocessing.verify_split import load_partition

PROCESSED_DIR = ROOT_DIR / "data" / "processed"
MODELS_DIR = ROOT_DIR / "models"
METRICS_DIR = ROOT_DIR / "results" / "metrics"

## 2. Load Processed Partitions

In [ ]:
try:
    X_train, y_train = load_partition(PROCESSED_DIR / "train", "train")
    X_val, y_val = load_partition(PROCESSED_DIR / "validation", "val")
    X_test, y_test = load_partition(PROCESSED_DIR / "test", "test")
    
    print(f"Train shape: {X_train.shape}, y_train: {y_train.shape}")
    print(f"Val shape:   {X_val.shape}, y_val:   {y_val.shape}")
    print(f"Test shape:  {X_test.shape}, y_test:  {y_test.shape}")
except FileNotFoundError as e:
    print("Processed partitions not found yet. Execute Phase 4 preprocessing pipeline once raw data is placed in data/raw/.")

## 3. Model Configuration & Training

In [ ]:
if 'X_train' in locals():
    xgb_model = XGBoostBaselineModel(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1
    )
    print("Training XGBoost on Train split ONLY (with validation monitoring)...")
    t0 = time.time()
    xgb_model.fit(X_train, y_train, eval_set=[(X_val, y_val)])
    t_train = time.time() - t0
    print(f"XGBoost model trained successfully in {t_train:.2f} seconds.")

## 4. Validation Set Evaluation

In [ ]:
if 'xgb_model' in locals() and xgb_model.model is not None:
    y_val_pred = xgb_model.predict(X_val)
    val_acc = accuracy_score(y_val, y_val_pred)
    val_f1_w = f1_score(y_val, y_val_pred, average='weighted', zero_division=0)
    val_f1_m = f1_score(y_val, y_val_pred, average='macro', zero_division=0)
    print(f"Validation Accuracy:    {val_acc*100:.2f}%")
    print(f"Validation Weighted F1: {val_f1_w:.4f}")
    print(f"Validation Macro F1:    {val_f1_m:.4f}")

## 5. Final Test Set Evaluation

In [ ]:
if 'xgb_model' in locals() and xgb_model.model is not None:
    y_test_pred = xgb_model.predict(X_test)
    test_acc = accuracy_score(y_test, y_test_pred)
    test_f1_w = f1_score(y_test, y_test_pred, average='weighted', zero_division=0)
    test_f1_m = f1_score(y_test, y_test_pred, average='macro', zero_division=0)
    print(f"Test Accuracy:    {test_acc*100:.2f}%")
    print(f"Test Weighted F1: {test_f1_w:.4f}")
    print(f"Test Macro F1:    {test_f1_m:.4f}")
    
    print("\nClassification Report (Test Set):")
    print(classification_report(y_test, y_test_pred, zero_division=0))

## 6. Confusion Matrix & Feature Importance

In [ ]:
if 'xgb_model' in locals() and xgb_model.model is not None:
    cm = confusion_matrix(y_test, y_test_pred)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Greens')
    plt.title('XGBoost — Test Confusion Matrix')
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.show()

## 7. Comparative Benchmark: Random Forest vs. XGBoost

In [ ]:
comp_file = METRICS_DIR / "model_comparison.csv"
if comp_file.exists():
    comp_df = pd.read_csv(comp_file)
    display(comp_df)
else:
    print("Model comparison table not populated yet.")